# 02 — Feature Engineering, Baseline Models, and Candidate Model Training

**Project:** Leadership Style Coach — University Edition  
**Author:** Romero Habib  
**Supervisor:** Dr. Rachad  
**Institution:** Lebanese American University (LAU), School of Arts and Sciences  

---

## Purpose

This notebook covers Phase 1 of the project pipeline (Weeks 1–2), specifically:

1. Loading and verifying the dataset
2. Feature engineering — encoding categorical variables and defining the feature matrix and target
3. Training and evaluating three required baseline classifiers: majority-class, logistic regression, and decision tree
4. Training and evaluating two candidate models: Random Forest and XGBoost
5. Comparing all five models against Hypothesis H1
6. Exporting the best-performing model and preprocessing pipeline for use in `03_explainability.ipynb` and the backend

## Research Hypothesis Being Tested

> **H1 — Classification Accuracy:** A supervised classification model trained on the Global Cultural Leadership Insights dataset can accurately classify a university student's dominant leadership style across four categories (Supportive, Transactional, Laissez-Faire, and Transformational), outperforming baseline classifiers including a majority-class classifier, logistic regression, and a decision tree.

## Primary Evaluation Metric

Due to class imbalance in the dataset (Supportive ≈ 40%, Transformational ≈ 14%), **macro-averaged F1 score** is used as the primary metric. This ensures that performance across all four classes is weighted equally, preventing the majority class from masking poor performance on minority classes. Accuracy is also reported for completeness.

---
## 1. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os
import pickle
import json

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)

# Output directories
os.makedirs('../models', exist_ok=True)
os.makedirs('../eda_outputs', exist_ok=True)

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('All libraries imported successfully.')
print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

---
## 2. Data Loading and Verification

The dataset is the **Global Cultural Leadership Insights** dataset (`CrossCultural_Leadership_Dataset_5k_2.csv`). Per the EDA completed in `01_eda.ipynb`, the dataset contains 6,945 rows, 21 columns, and zero missing values.

The target variable is `Leadership_Behavior_Encoded`, which encodes the four leadership style classes:  
- **0** = Laissez-Faire (1,411 instances)  
- **1** = Supportive (2,802 instances)  
- **2** = Transactional (1,775 instances)  
- **3** = Transformational (957 instances)

In [ ]:
df = pd.read_csv('../data/CrossCultural_Leadership_Dataset_5k_2.csv')

print(f'Dataset shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head(3)

In [ ]:
# Verify class distribution
class_counts = df['Preferred_Leadership_Behavior'].value_counts()
class_pct = df['Preferred_Leadership_Behavior'].value_counts(normalize=True) * 100

class_summary = pd.DataFrame({
    'Count': class_counts,
    'Percentage (%)': class_pct.round(1)
})
print('Target class distribution:')
print(class_summary)
print(f'\nMajority-class baseline accuracy (Supportive): {class_pct.max():.1f}%')

---
## 3. Feature Engineering

### 3.1 Feature Selection

The following feature groups are used as inputs to the model:

| Group | Features |
|---|---|
| **Behavioural (Likert 1–5)** | Role_Assumption, Production_Emphasis, Initiation_of_Structure, Tolerance_of_Uncertainty, Integration, Consideration |
| **Cultural (Hofstede, 0–100)** | Power_Distance, Individualism, Masculinity, Uncertainty_Avoidance, Long_Term_Orientation, Indulgence |
| **Demographics** | Age, Work_Experience_Years, Gender, Education_Level, Position_Level, Country |

**Key finding from EDA:** The six behavioural features carry almost all of the predictive signal. The cultural and demographic features show negligible class separation. All features are retained here so that SHAP (Notebook 03) can confirm this empirically — this is an expected and confirmable result.

`Respondent_ID` is dropped as it is an identifier with no predictive value.  
`Preferred_Leadership_Behavior` is dropped as it is the string form of the target.

In [ ]:
# Drop identifier and string target
df_model = df.drop(columns=['Respondent_ID', 'Preferred_Leadership_Behavior'])

# Define feature groups
BEHAVIOURAL_FEATURES = [
    'Role_Assumption', 'Production_Emphasis', 'Initiation_of_Structure',
    'Tolerance_of_Uncertainty', 'Integration', 'Consideration'
]

CULTURAL_FEATURES = [
    'Power_Distance', 'Individualism', 'Masculinity',
    'Uncertainty_Avoidance', 'Long_Term_Orientation', 'Indulgence'
]

NUMERIC_DEMO_FEATURES = ['Age', 'Work_Experience_Years']

CATEGORICAL_FEATURES = ['Country', 'Gender', 'Education_Level', 'Position_Level']

NUMERIC_FEATURES = BEHAVIOURAL_FEATURES + CULTURAL_FEATURES + NUMERIC_DEMO_FEATURES

TARGET = 'Leadership_Behavior_Encoded'

X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nNumeric features ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}')
print(f'\nCategorical features ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}')
print(f'\nTarget classes: {sorted(y.unique())} → {dict(zip(sorted(y.unique()), ["Laissez-Faire", "Supportive", "Transactional", "Transformational"]))}')

### 3.2 Train/Test Split

An 80/20 stratified split is used. Stratification ensures that the class distribution is preserved in both the training and test sets, which is important given the class imbalance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Training set: {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test set:     {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.1f}%)')
print()
print('Class distribution in training set:')
print(y_train.value_counts().sort_index())
print()
print('Class distribution in test set:')
print(y_test.value_counts().sort_index())

### 3.3 Preprocessing Pipeline

A `ColumnTransformer` is used to apply appropriate preprocessing to each feature type:

- **Numeric features** (behavioural, cultural, demographic): `StandardScaler` — required for logistic regression convergence; harmless for tree-based models
- **Categorical features** (Country, Gender, Education_Level, Position_Level): `OneHotEncoder` with `handle_unknown='ignore'` to handle unseen categories at inference time

This preprocessor is wrapped into each model's `Pipeline` so that feature encoding is reproducible and exportable as a single artifact.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES)
    ],
    remainder='drop'
)

# Fit on training data to inspect output shape
X_train_transformed = preprocessor.fit_transform(X_train)
print(f'Transformed training feature matrix shape: {X_train_transformed.shape}')

# Record feature names after one-hot encoding (needed for SHAP in notebook 03)
ohe_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(CATEGORICAL_FEATURES).tolist()
all_feature_names = NUMERIC_FEATURES + ohe_feature_names
print(f'Total features after encoding: {len(all_feature_names)}')
print(f'OHE features: {ohe_feature_names}')

---
## 4. Model Training and Evaluation

### Evaluation Helper

A shared evaluation function is defined to compute and display consistent metrics across all models.

In [ ]:
CLASS_NAMES = ['Laissez-Faire', 'Supportive', 'Transactional', 'Transformational']

results = {}  # Store all model results for comparison

def evaluate_model(name, pipeline, X_tr, y_tr, X_te, y_te, fit=True):
    """
    Fit (optional), predict, and evaluate a sklearn Pipeline.
    Stores results in the global `results` dict.
    Returns y_pred for downstream use.
    """
    if fit:
        pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)

    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_te, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_te, y_pred, average='macro', zero_division=0)

    results[name] = {
        'Accuracy': round(acc, 4),
        'Macro Precision': round(prec, 4),
        'Macro Recall': round(rec, 4),
        'Macro F1': round(f1, 4)
    }

    print(f'\n{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')
    print(f'  Accuracy        : {acc:.4f}')
    print(f'  Macro Precision : {prec:.4f}')
    print(f'  Macro Recall    : {rec:.4f}')
    print(f'  Macro F1        : {f1:.4f}  ← primary metric')
    print()
    print(classification_report(y_te, y_pred, target_names=CLASS_NAMES, zero_division=0))

    return y_pred

print('Evaluation function defined.')

---
### 4.1 Baseline 1 — Majority-Class Classifier

The majority-class classifier always predicts the most frequent class (Supportive, ≈40%). This is the simplest possible baseline and establishes the floor that any meaningful model must exceed. Because it never predicts the three minority classes, its macro F1 will be substantially lower than its accuracy.

In [ ]:
majority_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='most_frequent', random_state=42))
])

evaluate_model('Baseline: Majority-Class', majority_pipeline, X_train, y_train, X_test, y_test)

---
### 4.2 Baseline 2 — Logistic Regression

Logistic regression is a standard linear baseline. `class_weight='balanced'` is applied to account for class imbalance. `max_iter=1000` is set to ensure convergence on this dataset.

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        multi_class='multinomial',
        solver='lbfgs'
    ))
])

evaluate_model('Baseline: Logistic Regression', lr_pipeline, X_train, y_train, X_test, y_test)

---
### 4.3 Baseline 3 — Decision Tree

A decision tree provides an interpretable non-linear baseline. `class_weight='balanced'` is applied. `max_depth` is left unconstrained to allow the tree to fully grow, matching the default behaviour typically assumed for a decision tree baseline.

In [ ]:
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(
        class_weight='balanced',
        random_state=42
    ))
])

evaluate_model('Baseline: Decision Tree', dt_pipeline, X_train, y_train, X_test, y_test)

---
### 4.4 Candidate Model 1 — Random Forest

Random Forest is an ensemble of decision trees trained on random subsets of features and data. It is robust to overfitting relative to a single decision tree. `class_weight='balanced'` is applied to handle class imbalance. `n_estimators=300` provides a stable ensemble.

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_pred = evaluate_model('Candidate: Random Forest', rf_pipeline, X_train, y_train, X_test, y_test)

---
### 4.5 Candidate Model 2 — XGBoost

XGBoost is a gradient-boosted tree ensemble that has consistently achieved state-of-the-art results on tabular classification tasks (Skrynnyk & Vasylieva, 2022). `eval_metric='mlogloss'` is used for multi-class classification. `use_label_encoder=False` avoids a deprecation warning.

In [ ]:
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        eval_metric='mlogloss',
        use_label_encoder=False,
        random_state=42,
        n_jobs=-1
    ))
])

xgb_pred = evaluate_model('Candidate: XGBoost', xgb_pipeline, X_train, y_train, X_test, y_test)

---
## 5. Model Comparison and H1 Evaluation

### 5.1 Summary Table

All five models are compared across all four metrics. The **Macro F1** column is the primary metric for evaluating H1.

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('Macro F1', ascending=False)

# Highlight baselines vs candidates
results_df.index = [
    idx.replace('Baseline: ', '[Baseline] ').replace('Candidate: ', '[Candidate] ')
    for idx in results_df.index
]

print('Model Comparison — All Metrics (sorted by Macro F1):')
print()
print(results_df.to_string())

### 5.2 Bar Chart — Macro F1 Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = [
    '#2ecc71' if '[Candidate]' in idx else '#95a5a6'
    for idx in results_df.index
]

bars = ax.barh(results_df.index, results_df['Macro F1'], color=colors, edgecolor='white', height=0.5)

for bar, val in zip(bars, results_df['Macro F1']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Macro F1 Score', fontsize=11)
ax.set_title('Model Comparison — Macro F1 Score (Primary Metric)', fontsize=13, pad=12)
ax.set_xlim(0, 1.05)
ax.axvline(x=0.7, color='red', linestyle='--', alpha=0.4, label='0.70 reference')
ax.legend(fontsize=9)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='Candidate model'),
    Patch(facecolor='#95a5a6', label='Baseline model'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('../eda_outputs/model_comparison_macro_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_outputs/model_comparison_macro_f1.png')

### 5.3 Confusion Matrices — Candidate Models

Confusion matrices are generated for both candidate models to identify which leadership styles are most frequently confused with each other.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (pred, title) in zip(axes, [(rf_pred, 'Random Forest'), (xgb_pred, 'XGBoost')]):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix — {title}', fontsize=11, pad=10)
    ax.set_xticklabels(CLASS_NAMES, rotation=20, ha='right', fontsize=8)
    ax.set_yticklabels(CLASS_NAMES, fontsize=8)

plt.suptitle('Confusion Matrices — Candidate Models (Test Set)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../eda_outputs/confusion_matrices_candidates.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_outputs/confusion_matrices_candidates.png')

### 5.4 Cross-Validation — Best Candidate Model

5-fold stratified cross-validation is run on the best candidate model to confirm that test-set performance is representative and not an artifact of the specific train/test split.

In [ ]:
# Identify best candidate by Macro F1
candidate_results = {
    k: v for k, v in results.items() if 'Candidate' in k
}
best_name = max(candidate_results, key=lambda k: candidate_results[k]['Macro F1'])
best_pipeline = rf_pipeline if 'Random Forest' in best_name else xgb_pipeline

print(f'Best candidate model: {best_name}')
print(f'Test-set Macro F1: {candidate_results[best_name]["Macro F1"]:.4f}')
print()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_pipeline, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)

print(f'5-Fold Cross-Validation Macro F1 scores: {cv_scores.round(4)}')
print(f'Mean: {cv_scores.mean():.4f}  |  Std: {cv_scores.std():.4f}')

### 5.5 H1 Assessment

H1 requires the candidate model to outperform **all three** baselines across the primary metric (macro F1).

In [ ]:
baseline_names = [k for k in results if 'Baseline' in k]
best_candidate_f1 = candidate_results[best_name]['Macro F1']

print('H1 — Classification Accuracy Assessment')
print('=' * 55)
print(f'Best candidate ({best_name.replace("Candidate: ", "")}): Macro F1 = {best_candidate_f1:.4f}')
print()
all_passed = True
for bl in baseline_names:
    bl_f1 = results[bl]['Macro F1']
    passed = best_candidate_f1 > bl_f1
    status = 'PASS ✓' if passed else 'FAIL ✗'
    if not passed:
        all_passed = False
    print(f'  vs {bl.replace("Baseline: ", ""):30s}: {bl_f1:.4f}  →  {status}')

print()
if all_passed:
    print('H1 SUPPORTED: The candidate model outperforms all three baseline classifiers on macro F1.')
else:
    print('H1 NOT FULLY SUPPORTED: The candidate model does not outperform all baselines.')

---
## 6. Model Export

The best-performing model pipeline (preprocessor + classifier) is exported as `trained_model.pkl`. A metadata file is also saved containing the feature names, class labels, and model name — required by the backend inference engine and by `03_explainability.ipynb`.

### What is exported

| File | Contents | Used by |
|---|---|---|
| `models/trained_model.pkl` | Full sklearn Pipeline (preprocessor + classifier) | Backend, Notebook 03 |
| `models/feature_names.json` | Ordered list of all feature names after OHE | Backend, SHAP |
| `models/class_labels.json` | Mapping from encoded integer → class name | Backend, Frontend |
| `models/model_metadata.json` | Model name, metrics, feature groups | Backend, Report |

In [ ]:
# Export the best model pipeline
model_path = '../models/trained_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(best_pipeline, f)
print(f'Exported: {model_path}')

# Feature names (post-OHE, matching preprocessor output order)
feature_names_path = '../models/feature_names.json'
with open(feature_names_path, 'w') as f:
    json.dump(all_feature_names, f, indent=2)
print(f'Exported: {feature_names_path}')

# Class label mapping
class_labels = {
    0: 'Laissez-Faire',
    1: 'Supportive',
    2: 'Transactional',
    3: 'Transformational'
}
class_labels_path = '../models/class_labels.json'
with open(class_labels_path, 'w') as f:
    json.dump(class_labels, f, indent=2)
print(f'Exported: {class_labels_path}')

# Model metadata
metadata = {
    'model_name': best_name.replace('Candidate: ', ''),
    'test_accuracy': candidate_results[best_name]['Accuracy'],
    'test_macro_f1': candidate_results[best_name]['Macro F1'],
    'test_macro_precision': candidate_results[best_name]['Macro Precision'],
    'test_macro_recall': candidate_results[best_name]['Macro Recall'],
    'cv_macro_f1_mean': round(float(cv_scores.mean()), 4),
    'cv_macro_f1_std': round(float(cv_scores.std()), 4),
    'train_size': int(X_train.shape[0]),
    'test_size': int(X_test.shape[0]),
    'n_features_input': int(X.shape[1]),
    'n_features_transformed': int(X_train_transformed.shape[1]),
    'feature_groups': {
        'behavioural': BEHAVIOURAL_FEATURES,
        'cultural': CULTURAL_FEATURES,
        'numeric_demographic': NUMERIC_DEMO_FEATURES,
        'categorical_demographic': CATEGORICAL_FEATURES
    },
    'target_classes': class_labels,
    'h1_supported': bool(all_passed)
}
metadata_path = '../models/model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Exported: {metadata_path}')

print()
print('All model artifacts exported to models/')

### 6.1 Verify Export

In [ ]:
# Reload and verify the exported model produces identical predictions
with open('../models/trained_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

reload_pred = loaded_model.predict(X_test)
reload_f1 = f1_score(y_test, reload_pred, average='macro')

assert np.array_equal(reload_pred, best_pipeline.predict(X_test)), 'Prediction mismatch after reload!'
print(f'Export verified. Reloaded model macro F1: {reload_f1:.4f} (matches original)')

---
## 7. Summary

This notebook completed the following steps for Phase 1 of the Leadership Style Coach project:

| Step | Outcome |
|---|---|
| Dataset loaded | 6,945 rows, 21 columns, zero missing values |
| Feature engineering | 14 numeric + 4 categorical → standardised + one-hot encoded |
| Train/test split | 80/20 stratified (5,556 train / 1,389 test) |
| Majority-class baseline | Macro F1 reported |
| Logistic regression baseline | Macro F1 reported |
| Decision tree baseline | Macro F1 reported |
| Random Forest candidate | Macro F1 reported |
| XGBoost candidate | Macro F1 reported |
| H1 assessment | See Section 5.5 |
| Model exported | `models/trained_model.pkl` |

## Next Step

Proceed to `03_explainability.ipynb`, which loads `trained_model.pkl` and applies:
1. **SHAP** — global and per-student feature importance (H2)
2. **Counterfactual explanations** — minimum profile changes to reach a different classification (H2)
3. **Consistency metric** — cosine/Euclidean similarity analysis across student pairs